In [2]:
from pathlib import Path

print("Thư mục hiện tại:", Path.cwd())

for root in [Path("/kaggle/working"), Path("/kaggle/input")]:
    print(f"\n=== {root} ===")
    if not root.exists():
        print("Không tồn tại")
        continue

    entries = list(root.iterdir())
    if not entries:
        print("(Trống)")
    else:
        for p in entries:
            print("[DIR]" if p.is_dir() else "[FILE]", p.name)

Thư mục hiện tại: /kaggle/working

=== /kaggle/working ===
[DIR] .virtual_documents
[DIR] Speech-to-Text-PhoWhisper

=== /kaggle/input ===
[DIR] datasets


In [3]:
from pathlib import Path

repo = Path("/kaggle/working/Speech-to-Text-PhoWhisper")

print("Repo tồn tại:", repo.exists())

if repo.exists():
    print("\nCác file/thư mục trong repo:")
    for p in repo.iterdir():
        print("[DIR]" if p.is_dir() else "[FILE]", p.name)

    print(
        "\nCó script fine-tuning:",
        (repo / "kaggle_training" / "train.py").exists()
    )
else:
    print("Không thấy repo ở đường dẫn này; kiểm tra output cell trước.")

Repo tồn tại: True

Các file/thư mục trong repo:
[DIR] kaggle_training
[DIR] models
[DIR] benchmarks
[DIR] .git
[DIR] tests
[FILE] README.md
[FILE] requirements.txt
[DIR] tools
[FILE] pyproject.toml
[FILE] .gitignore
[DIR] outputs
[DIR] src

Có script fine-tuning: True


In [4]:
%cd /kaggle/working/Speech-to-Text-PhoWhisper

/kaggle/working/Speech-to-Text-PhoWhisper


In [5]:
from pathlib import Path
from itertools import islice

for dataset_dir in Path("/kaggle/input").iterdir():
    print(f"\nINPUT: {dataset_dir}")

    if dataset_dir.is_dir():
        for p in islice(dataset_dir.iterdir(), 15):
            print(" ", "[DIR]" if p.is_dir() else "[FILE]", p.name)


INPUT: /kaggle/input/datasets
  [DIR] tuandaodev


In [7]:
from pathlib import Path
import pandas as pd

DATA_ROOT = Path(
    "/kaggle/input/datasets/tuandaodev/viet-bud500-formatted"
)

metadata_path = DATA_ROOT / "metadata_train.csv"

# Chỉ đọc vài dòng để xác nhận format.
preview = pd.read_csv(metadata_path, sep="|", nrows=5)

print("Các cột:", preview.columns.tolist())
print(preview.to_string(index=False))

print("\nMột vài tên audio:")
from itertools import islice
for p in islice((DATA_ROOT / "wavs").iterdir(), 5):
    print(p.name)

Các cột: ['audio_file', 'text', 'speaker_name']
          audio_file                                    text speaker_name
wavs/audio141889.wav  đưa tay bóp đầu của mình vũ thành quân     viet_bud
wavs/audio573174.wav tôi đau ốm liên miên làm cho hai ông bà     viet_bud
wavs/audio573170.wav chỉ nhìn về phía hạ hạ thỉnh thoảng lén     viet_bud
wavs/audio141909.wav  ngày càng phát triển dường như cái tết     viet_bud
wavs/audio573172.wav tiết địa kiểm tra vào tiết đầu hay tiết     viet_bud

Một vài tên audio:
audio66809.wav
audio111207.wav
audio145828.wav
audio574256.wav
audio258548.wav


In [8]:
from pathlib import Path
import pandas as pd

DATA_ROOT = Path(
    "/kaggle/input/datasets/tuandaodev/viet-bud500-formatted"
)
OUTPUT = Path("/kaggle/working/audio200-preview")
OUTPUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(
    DATA_ROOT / "metadata_train.csv",
    sep="|",
    dtype=str,
    keep_default_na=False,
)

required = {"audio_file", "text", "speaker_name"}
assert required.issubset(df.columns), f"Thiếu cột: {required - set(df.columns)}"

for column in required:
    df[column] = df[column].str.strip()

print("Tổng dòng metadata train:", len(df))
print("Số speaker_name khác nhau:", df["speaker_name"].nunique())
print("\nCác speaker_name phổ biến:")
print(df["speaker_name"].value_counts().head(10).to_string())

# Loại đường dẫn/nhãn rỗng và audio_file lặp.
eligible = df[
    df["audio_file"].ne("") & df["text"].ne("")
].drop_duplicates(subset=["audio_file"]).copy()

audio_ids = pd.to_numeric(
    eligible["audio_file"].str.extract(
        r"(?:^|/)audio(\d+)\.wav$", expand=False
    ),
    errors="coerce",
)
eligible = eligible.loc[audio_ids.gt(1250)]

# Trộn có seed cố định, rồi chỉ kiểm tra sự tồn tại
# của các file ứng viên cho tới khi đủ 200.
selected_indices = []
missing = 0

for index, row in eligible.sample(frac=1, random_state=42).iterrows():
    relative_path = Path(row["audio_file"])

    if relative_path.is_absolute() or ".." in relative_path.parts:
        raise ValueError(f"Đường dẫn không hợp lệ: {relative_path}")

    if (DATA_ROOT / relative_path).is_file():
        selected_indices.append(index)
        if len(selected_indices) == 200:
            break
    else:
        missing += 1

selected = eligible.loc[selected_indices].copy()
assert len(selected) == 200, f"Chỉ tìm được {len(selected)} audio hợp lệ"

destination = OUTPUT / "selected_train_200.csv"
selected.to_csv(destination, sep="|", index=False)

print("\nĐã chọn:", len(selected), "audio")
print("File không tồn tại gặp trong lúc chọn:", missing)
print("Số speaker trong 200 mẫu:", selected["speaker_name"].nunique())
print("Danh sách lưu tại:", destination)
print("\n5 mẫu đầu:")
print(selected.head().to_string(index=False))

Tổng dòng metadata train: 634158
Số speaker_name khác nhau: 1

Các speaker_name phổ biến:
speaker_name
viet_bud    634158

Đã chọn: 200 audio
File không tồn tại gặp trong lúc chọn: 0
Số speaker trong 200 mẫu: 1
Danh sách lưu tại: /kaggle/working/audio200-preview/selected_train_200.csv

5 mẫu đầu:
          audio_file                                    text speaker_name
wavs/audio310424.wav   về sau ra mình có thể đã có nghề chưa     viet_bud
wavs/audio123226.wav  vũ nhanh chân tiến lại gần chỗ cô ngồi     viet_bud
wavs/audio270087.wav   vào bệnh viện bác sĩ khám cho tôi một     viet_bud
wavs/audio222706.wav  việc hãy để cô ấy ngày ngày đi làm tối     viet_bud
wavs/audio135543.wav còn phải chen chúc lê la trong một ngôi     viet_bud


In [9]:
%cd /kaggle/working/Speech-to-Text-PhoWhisper
!pip install -r kaggle_training/requirements.txt

/kaggle/working/Speech-to-Text-PhoWhisper
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 82.9 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 90.1 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.2/333.2 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.5/447.5 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 82.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 435.4/435.4 kB 23.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 44.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from pathlib import Path
import sys
import subprocess

print("=== SESSION ===")
print("Python:", sys.version)
print("Python executable:", sys.executable)
print("Thư mục hiện tại:", Path.cwd())

for root in [Path("/kaggle/input"), Path("/kaggle/working")]:
    print(f"\n=== {root} ===")
    if not root.exists():
        print("Không tồn tại")
        continue

    entries = list(root.iterdir())
    if not entries:
        print("(Trống)")
    for p in entries:
        print("[DIR]" if p.is_dir() else "[FILE]", p.name)

print("\n=== SOURCE CODE VÀ FILE THỬ NGHIỆM ===")
checks = {
    "Script training": Path(
        "/kaggle/working/Speech-to-Text-PhoWhisper/kaggle_training/train.py"
    ),
    "Dependencies training": Path(
        "/kaggle/working/Speech-to-Text-PhoWhisper/kaggle_training/requirements.txt"
    ),
    "Danh sách 200 audio": Path(
        "/kaggle/working/audio200-preview/selected_train_200.csv"
    ),
    "Manifest train": Path(
        "/kaggle/working/audio200-manifests/train.json"
    ),
    "Manifest validation": Path(
        "/kaggle/working/audio200-manifests/validation.json"
    ),
}

for label, path in checks.items():
    print(f"{label}: {'CÓ' if path.is_file() else 'KHÔNG THẤY'}")

print("\n=== NVIDIA GPU ===")
try:
    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,driver_version",
            "--format=csv",
        ],
        capture_output=True,
        text=True,
        timeout=20,
    )
    print(result.stdout or result.stderr)
except (FileNotFoundError, subprocess.TimeoutExpired) as error:
    print(type(error).__name__, str(error))

=== SESSION ===
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
Python executable: /usr/bin/python3
Thư mục hiện tại: /kaggle/working

=== /kaggle/input ===
(Trống)

=== /kaggle/working ===
[DIR] .virtual_documents

=== SOURCE CODE VÀ FILE THỬ NGHIỆM ===
Script training: KHÔNG THẤY
Dependencies training: KHÔNG THẤY
Danh sách 200 audio: KHÔNG THẤY
Manifest train: KHÔNG THẤY
Manifest validation: KHÔNG THẤY

=== NVIDIA GPU ===
name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.178.04
Tesla T4, 15360 MiB, 580.178.04



In [3]:
from pathlib import Path
import os
import csv

BASE = Path("/kaggle/input")
metadata_paths = []

MAX_DEPTH = 5
MAX_DIRECTORIES = 40
visited = 0

print("=== TÌM METADATA TRONG INPUT HIỆN TẠI ===")

for current, dirs, files in os.walk(BASE):
    current = Path(current)
    depth = len(current.relative_to(BASE).parts)

    # Không đi vào thư mục audio.
    dirs[:] = sorted(
        d for d in dirs
        if d.lower() not in {"wavs", "wav", "audio", "audios"}
    )

    print(current)

    for name in files:
        if name.lower().endswith((".csv", ".tsv", ".json", ".jsonl", ".parquet")):
            metadata_paths.append(current / name)

    if depth >= MAX_DEPTH:
        dirs[:] = []

    visited += 1
    if visited >= MAX_DIRECTORIES:
        print("Dừng ở giới hạn số thư mục kiểm tra.")
        break

if not metadata_paths:
    print("\nKhông tìm thấy metadata trong phạm vi kiểm tra.")
    print("Nếu Input trống thì hiện tại chưa có dataset được gắn.")
else:
    print("\nCác metadata tìm thấy:")
    for path in metadata_paths[:20]:
        print(path, "|", path.stat().st_size, "bytes")

    # Chỉ xem vài dòng của tối đa 2 CSV train.
    train_csvs = [
        p for p in metadata_paths
        if p.suffix.lower() == ".csv" and "train" in p.name.lower()
    ]

    for path in train_csvs[:2]:
        print(f"\n=== PREVIEW: {path} ===")

        with path.open(encoding="utf-8-sig", newline="") as f:
            sample = f.read(8192)
            f.seek(0)

            try:
                dialect = csv.Sniffer().sniff(sample, delimiters="|,\t;")
            except csv.Error:
                print("Chưa xác định được delimiter. Nội dung đầu:")
                print(sample[:1500])
                continue

            reader = csv.DictReader(f, dialect=dialect)
            print("Delimiter:", repr(dialect.delimiter))
            print("Các cột:", reader.fieldnames)

            for i, row in enumerate(reader):
                if i >= 3:
                    break
                print(row)

                # Nếu schema đã biết, kiểm tra đúng vài đường dẫn mẫu.
                name = row.get("audio_file")
                if name:
                    relative = Path(name)
                    if not relative.is_absolute() and ".." not in relative.parts:
                        audio_path = path.parent / relative
                        print("Audio mẫu tồn tại:", audio_path.is_file())

=== TÌM METADATA TRONG INPUT HIỆN TẠI ===
/kaggle/input

Không tìm thấy metadata trong phạm vi kiểm tra.
Nếu Input trống thì hiện tại chưa có dataset được gắn.


In [4]:
from importlib.metadata import version, PackageNotFoundError
import subprocess
import sys

print("=== PHIÊN BẢN ĐÃ CÀI ===")

packages = [
    "torch",
    "torchvision",
    "numpy",
    "scipy",
    "soundfile",
    "transformers",
    "accelerate",
    "huggingface-hub",
    "tokenizers",
    "safetensors",
    "jiwer",
]

for package in packages:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: CHƯA CÀI")

print("\n=== IMPORT VÀ CUDA TRONG PROCESS RIÊNG ===")

check_code = """
import sys
print("Python:", sys.version, flush=True)

import numpy
import scipy
import soundfile
import torch
import accelerate
import huggingface_hub
import jiwer

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

print("PASS: import các thành phần training.", flush=True)
print("CUDA available:", torch.cuda.is_available(), flush=True)
print("GPU count:", torch.cuda.device_count(), flush=True)

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print("GPU", i, torch.cuda.get_device_name(i), flush=True)
    x = torch.ones(4, device="cuda")
    print("CUDA calculation:", (x + 1).cpu().tolist(), flush=True)
else:
    print("Chưa thể chạy GPU smoke test.", flush=True)
"""

try:
    result = subprocess.run(
        [sys.executable, "-c", check_code],
        capture_output=True,
        text=True,
        timeout=120,
    )

    print("Exit code:", result.returncode)
    print(result.stdout)

    if result.stderr:
        print("=== STDERR / TRACEBACK ===")
        print(result.stderr)
except subprocess.TimeoutExpired:
    print("Kiểm tra import vượt quá 120 giây; chưa xác nhận thành công.")

=== PHIÊN BẢN ĐÃ CÀI ===
torch: 2.11.0+cu128
torchvision: 0.26.0+cu128
numpy: 2.1.3
scipy: 1.16.3
soundfile: 0.14.0
transformers: 5.16.1
accelerate: 1.14.0
huggingface-hub: 1.29.0
tokenizers: 0.23.1
safetensors: 0.8.0
jiwer: CHƯA CÀI

=== IMPORT VÀ CUDA TRONG PROCESS RIÊNG ===
Exit code: 1
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]

=== STDERR / TRACEBACK ===
Traceback (most recent call last):
  File "<string>", line 11, in <module>
    import jiwer
ModuleNotFoundError: No module named 'jiwer'



In [6]:
from pathlib import Path
import subprocess

REPO = Path("/kaggle/working/Speech-to-Text-PhoWhisper")

if not REPO.exists():
    subprocess.run(
        [
            "git", "clone",
            "https://github.com/apase95/Speech-to-Text-PhoWhisper.git",
            str(REPO),
        ],
        check=True,
    )
else:
    print("Repo đã tồn tại, không clone lại.")

print(
    "Có script training:",
    (REPO / "kaggle_training/train.py").is_file()
)

Repo đã tồn tại, không clone lại.
Có script training: True


In [7]:
import shutil
import subprocess

kaggle_cli = shutil.which("kaggle")
print("Kaggle CLI:", kaggle_cli)

if kaggle_cli:
    commands = [
        ["kaggle", "--version"],
        ["kaggle", "datasets", "files", "--help"],
        ["kaggle", "datasets", "download", "--help"],
    ]

    for command in commands:
        print("\nCOMMAND:", " ".join(command))
        result = subprocess.run(
            command,
            capture_output=True,
            text=True,
            timeout=60,
        )
        print("Exit code:", result.returncode)
        print(result.stdout)
        if result.stderr:
            print(result.stderr)
else:
    print("Chưa có CLI; chưa cài thêm gì ở bước này.")

Kaggle CLI: /usr/local/bin/kaggle

COMMAND: kaggle --version
Exit code: 0
Kaggle CLI 2.0.2


COMMAND: kaggle datasets files --help
Exit code: 0
usage: kaggle datasets files [-h] [-v] [--page-token PAGE_TOKEN]
                             [--page-size PAGE_SIZE]
                             [dataset]

options:
  -h, --help            show this help message and exit
  dataset               Dataset URL suffix in format <owner>/<dataset-name> (use "kaggle datasets list" to show options)
  -v, --csv             Print results in CSV format (if not set print in table format)
  --page-token PAGE_TOKEN
                        Page token for results paging.
  --page-size PAGE_SIZE
                        Number of items to show on a page. Default size is 20, max is 200


COMMAND: kaggle datasets download --help
Exit code: 0
usage: kaggle datasets download [-h] [-f FILE_NAME] [-p PATH] [-w] [--unzip]
                                [-o] [-q]
                                [dataset]

options:
  -

In [8]:
import subprocess

result = subprocess.run(
    [
        "kaggle", "datasets", "files",
        "tuandaodev/viet-bud500-formatted",
    ],
    capture_output=True,
    text=True,
    timeout=120,
)

print("Exit code:", result.returncode)
print(result.stdout)

if result.stderr:
    print("STDERR:")
    print(result.stderr)

Exit code: 0
Next Page Token = CfDJ8JDwKpgsc5hJpcljKjpYpOGY39rcy8M-b5G6h6_DkdRw1AjkQroekJ43u0dm9fbtI01i7pPiq6PQFbtafe7RQKQlszI97D1c5XdAQ1jYEDHH24Yiwm-0ImWnnnubnw0LecOWb6-RBA0G7EXY
name                      size  creationDate                
--------------------  --------  --------------------------  
metadata_test.csv       606949  2024-10-25 13:33:36.989000  
metadata_train.csv    52471990  2024-10-25 13:33:39.816000  
wavs/audio1.wav         132424  2024-10-25 13:59:05.444000  
wavs/audio10.wav         95300  2024-10-25 13:43:26.190000  
wavs/audio100.wav       128680  2024-10-25 13:43:22.459000  
wavs/audio1000.wav       85678  2024-10-25 13:57:55.502000  
wavs/audio10000.wav     157082  2024-10-25 13:47:14.894000  
wavs/audio100000.wav     98838  2024-10-25 13:57:25.900000  
wavs/audio100001.wav    103238  2024-10-25 13:57:24.372000  
wavs/audio100002.wav    110356  2024-10-25 13:44:33.774000  
wavs/audio100003.wav    135896  2024-10-25 13:50:50.749000  
wavs/audio100004.wav     87

In [9]:
from pathlib import Path
import subprocess

DATASET = "tuandaodev/viet-bud500-formatted"
SUBSET = Path("/kaggle/working/vietbud200")
SUBSET.mkdir(parents=True, exist_ok=True)

command = [
    "kaggle", "datasets", "download",
    DATASET,
    "-f", "metadata_train.csv",
    "-p", str(SUBSET),
]

print("COMMAND:", " ".join(command))
subprocess.run(command, check=True, timeout=300)

print("\nCác file đã nhận:")
for p in SUBSET.iterdir():
    if p.is_file():
        print(p.name, "|", p.stat().st_size, "bytes")

COMMAND: kaggle datasets download tuandaodev/viet-bud500-formatted -f metadata_train.csv -p /kaggle/working/vietbud200
Dataset URL: https://www.kaggle.com/datasets/tuandaodev/viet-bud500-formatted
License(s): CC0-1.0


100%|██████████| 50.0M/50.0M [00:03<00:00, 13.7MB/s]




Các file đã nhận:
metadata_train.csv | 52471990 bytes


In [10]:
from pathlib import Path
import csv
import random
import re
import zipfile

SUBSET = Path("/kaggle/working/vietbud200")
metadata_path = SUBSET / "metadata_train.csv"

# Nếu API trả về ZIP, chỉ lấy đúng metadata cần dùng.
if not metadata_path.is_file():
    found = False

    for archive in SUBSET.glob("*.zip"):
        with zipfile.ZipFile(archive) as z:
            matches = [
                name for name in z.namelist()
                if Path(name).name == "metadata_train.csv"
            ]

            if len(matches) == 1:
                with z.open(matches[0]) as source:
                    metadata_path.write_bytes(source.read())
                found = True
                break

    assert found, "Không tìm thấy metadata_train.csv trong file đã tải."

with metadata_path.open(encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f, delimiter="|")

    required = {"audio_file", "text", "speaker_name"}
    assert required.issubset(reader.fieldnames or []), (
        f"Schema không đúng: {reader.fieldnames}"
    )

    rows = list(reader)

print("Tổng dòng metadata:", len(rows))
print("Các cột:", reader.fieldnames)

# Chọn ngẫu nhiên có seed cố định.
# Không truy cập audio vì chúng chưa được tải.
eligible = []
seen = set()

for row in rows:
    name = row["audio_file"].strip()
    text = row["text"].strip()

    match = re.fullmatch(r"(?:wavs/)?audio(\d+)\.wav", name)

    if not match or not text:
        continue

    # Giữ quy tắc loại ID thấp của pipeline hiện tại.
    if int(match.group(1)) <= 1250:
        continue

    normalized_name = f"wavs/audio{match.group(1)}.wav"

    if normalized_name in seen:
        raise ValueError(f"Đường dẫn audio trùng trong metadata: {normalized_name}")

    seen.add(normalized_name)
    eligible.append({
        "audio_file": normalized_name,
        "text": text,
        "speaker_name": row["speaker_name"].strip(),
    })

assert len(eligible) >= 200, "Không đủ 200 mẫu ứng viên."

selected = random.Random(42).sample(eligible, 200)

selected_path = SUBSET / "selected_train_200.csv"
with selected_path.open("w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["audio_file", "text", "speaker_name"],
        delimiter="|",
    )
    writer.writeheader()
    writer.writerows(selected)

print("Đã chọn:", len(selected), "mẫu")
print("Danh sách:", selected_path)

print("\n3 mẫu đầu:")
for row in selected[:3]:
    print(row)

Tổng dòng metadata: 634158
Các cột: ['audio_file', 'text', 'speaker_name']
Đã chọn: 200 mẫu
Danh sách: /kaggle/working/vietbud200/selected_train_200.csv

3 mẫu đầu:
{'audio_file': 'wavs/audio562383.wav', 'text': 'mình lại phải lựa chọn nhiều hơn nữa là', 'speaker_name': 'viet_bud'}
{'audio_file': 'wavs/audio246240.wav', 'text': 'của họ chưa bao giờ được bảo trì hoặc', 'speaker_name': 'viet_bud'}
{'audio_file': 'wavs/audio296430.wav', 'text': 'ra ngoài cà phê nhìn anh tiều tụy hai', 'speaker_name': 'viet_bud'}


In [11]:
from pathlib import Path
import csv
import subprocess

SUBSET = Path("/kaggle/working/vietbud200")
PROBE = SUBSET / "download-probe"
PROBE.mkdir(exist_ok=True)

with (SUBSET / "selected_train_200.csv").open(
    encoding="utf-8", newline=""
) as f:
    first = next(csv.DictReader(f, delimiter="|"))

command = [
    "kaggle", "datasets", "download",
    "tuandaodev/viet-bud500-formatted",
    "-f", first["audio_file"],
    "-p", str(PROBE),
]

print("Audio được yêu cầu:", first["audio_file"])
print("COMMAND:", " ".join(command))
subprocess.run(command, check=True, timeout=180)

print("\nCác file thực tế nhận được:")
for p in PROBE.rglob("*"):
    if p.is_file():
        print(p.relative_to(PROBE), "|", p.stat().st_size, "bytes")

Audio được yêu cầu: wavs/audio562383.wav
COMMAND: kaggle datasets download tuandaodev/viet-bud500-formatted -f wavs/audio562383.wav -p /kaggle/working/vietbud200/download-probe
Dataset URL: https://www.kaggle.com/datasets/tuandaodev/viet-bud500-formatted
License(s): CC0-1.0


  0%|          | 0.00/90.5k [00:00<?, ?B/s]



Các file thực tế nhận được:
audio562383.wav | 92654 bytes


100%|██████████| 90.5k/90.5k [00:00<00:00, 347kB/s]


In [12]:
import csv
import os
import re
import shutil
import subprocess
import tempfile
import time
import wave
from pathlib import Path

ROOT = Path("/kaggle/working/vietbud200")
CSV_PATH = ROOT / "selected_train_200.csv"
WAV_DIR = ROOT / "subset" / "wavs"
STAGING = ROOT / "subset" / ".staging"
PROBE = ROOT / "download-probe" / "audio562383.wav"

DATASET = "tuandaodev/viet-bud500-formatted"

def validate_wav(path):
    with wave.open(str(path), "rb") as audio:
        frames = audio.getnframes()
        rate = audio.getframerate()
        channels = audio.getnchannels()
        width = audio.getsampwidth()

        if min(frames, rate, channels, width) <= 0:
            raise ValueError(f"Audio rỗng/không hợp lệ: {path.name}")

        if frames / rate > 30:
            raise ValueError(f"Audio dài hơn 30 giây: {path.name}")

        remaining = frames
        while remaining:
            count = min(remaining, 8192)
            block = audio.readframes(count)
            if len(block) != count * channels * width:
                raise ValueError(f"WAV bị cắt cụt: {path.name}")
            remaining -= count

def is_valid(path):
    if not path.is_file() or path.is_symlink():
        return False
    try:
        validate_wav(path)
        return True
    except (OSError, EOFError, wave.Error, ValueError):
        return False

with CSV_PATH.open(encoding="utf-8-sig", newline="") as f:
    rows = list(csv.DictReader(f, delimiter="|"))

names = [row["audio_file"] for row in rows]

assert len(names) == 200, f"Cần 200 mẫu, nhận {len(names)}"
assert len(set(names)) == 200, "Danh sách có audio trùng"
assert all(
    re.fullmatch(r"wavs/audio\d+\.wav", name)
    for name in names
), "Có đường dẫn audio không hợp lệ"
assert shutil.which("kaggle"), "Không tìm thấy Kaggle CLI"

WAV_DIR.mkdir(parents=True, exist_ok=True)
STAGING.mkdir(parents=True, exist_ok=True)

counts = {"existing": 0, "probe": 0, "downloaded": 0}
print("Bắt đầu tải 200 audio vào:", WAV_DIR, flush=True)

for index, remote_name in enumerate(names, 1):
    basename = Path(remote_name).name
    destination = WAV_DIR / basename

    if is_valid(destination):
        counts["existing"] += 1

    elif basename == PROBE.name and is_valid(PROBE):
        with tempfile.TemporaryDirectory(dir=STAGING) as tmp:
            staged = Path(tmp) / basename
            shutil.copyfile(PROBE, staged)
            validate_wav(staged)
            os.replace(staged, destination)
        counts["probe"] += 1

    else:
        completed = False
        last_error = ""

        for attempt in range(1, 4):
            try:
                with tempfile.TemporaryDirectory(dir=STAGING) as tmp:
                    result = subprocess.run(
                        [
                            "kaggle", "datasets", "download",
                            DATASET,
                            "-f", remote_name,
                            "-p", tmp,
                            "--force",
                            "--quiet",
                        ],
                        capture_output=True,
                        text=True,
                        timeout=180,
                    )

                    if result.returncode != 0:
                        raise RuntimeError(
                            result.stderr or result.stdout
                            or f"CLI exit code {result.returncode}"
                        )

                    staged = Path(tmp) / basename
                    assert staged.is_file(), (
                        f"Không nhận được file dự kiến: {basename}"
                    )
                    validate_wav(staged)
                    os.replace(staged, destination)

                counts["downloaded"] += 1
                completed = True
                break

            except (
                OSError, EOFError, wave.Error, ValueError,
                RuntimeError, AssertionError,
                subprocess.TimeoutExpired,
            ) as error:
                last_error = str(error)[-1000:]
                if attempt < 3:
                    time.sleep(2 * attempt)

        if not completed:
            print(f"DỪNG tại {index}/200: {remote_name}", flush=True)
            print("Thống kê:", counts, flush=True)
            raise RuntimeError(last_error)

    if index % 10 == 0:
        print(f"Tiến độ: {index}/200 | {counts}", flush=True)

total_bytes = sum(
    (WAV_DIR / Path(name).name).stat().st_size
    for name in names
)

print("\nHOÀN TẤT: 200/200 audio đã kiểm tra.")
print("Thống kê:", counts)
print("Dung lượng audio:", round(total_bytes / 1024**2, 2), "MiB")
print("Thư mục:", WAV_DIR)

Bắt đầu tải 200 audio vào: /kaggle/working/vietbud200/subset/wavs
Tiến độ: 10/200 | {'existing': 0, 'probe': 1, 'downloaded': 9}
Tiến độ: 20/200 | {'existing': 0, 'probe': 1, 'downloaded': 19}
Tiến độ: 30/200 | {'existing': 0, 'probe': 1, 'downloaded': 29}
Tiến độ: 40/200 | {'existing': 0, 'probe': 1, 'downloaded': 39}
Tiến độ: 50/200 | {'existing': 0, 'probe': 1, 'downloaded': 49}
Tiến độ: 60/200 | {'existing': 0, 'probe': 1, 'downloaded': 59}
Tiến độ: 70/200 | {'existing': 0, 'probe': 1, 'downloaded': 69}
Tiến độ: 80/200 | {'existing': 0, 'probe': 1, 'downloaded': 79}
Tiến độ: 90/200 | {'existing': 0, 'probe': 1, 'downloaded': 89}
Tiến độ: 100/200 | {'existing': 0, 'probe': 1, 'downloaded': 99}
Tiến độ: 110/200 | {'existing': 0, 'probe': 1, 'downloaded': 109}
Tiến độ: 120/200 | {'existing': 0, 'probe': 1, 'downloaded': 119}
Tiến độ: 130/200 | {'existing': 0, 'probe': 1, 'downloaded': 129}
Tiến độ: 140/200 | {'existing': 0, 'probe': 1, 'downloaded': 139}
Tiến độ: 150/200 | {'existing'

In [14]:
import subprocess
import sys

check_code = r"""
import sys
print("Python:", sys.version, flush=True)

import numpy as np
import scipy
import soundfile as sf
import torch
import transformers
import accelerate
import jiwer

from transformers import (
    WhisperProcessor,
    WhisperForConditionalGeneration,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)

print("PASS: import các class training.", flush=True)
print("NumPy:", np.__version__, flush=True)
print("Transformers:", transformers.__version__, flush=True)
print("Accelerate:", accelerate.__version__, flush=True)
print("Torch:", torch.__version__, flush=True)

assert torch.cuda.is_available(), "CUDA chưa sẵn sàng"
print("Số GPU:", torch.cuda.device_count(), flush=True)

for i in range(torch.cuda.device_count()):
    print("GPU", i, torch.cuda.get_device_name(i), flush=True)

# Chỉ kiểm tra CUDA/autograd bằng một tensor nhỏ.
x = torch.ones(4, device="cuda", requires_grad=True)
loss = (x * x).mean()
loss.backward()
torch.cuda.synchronize()

print("CUDA loss:", loss.item(), flush=True)
print("CUDA gradient:", x.grad.cpu().tolist(), flush=True)
print("PASS: CUDA forward/backward.", flush=True)
"""

result = subprocess.run(
    [sys.executable, "-c", check_code],
    capture_output=True,
    text=True,
    timeout=120,
)

print("Exit code:", result.returncode)
print(result.stdout)

if result.stderr:
    print("=== STDERR / TRACEBACK ===")
    print(result.stderr)

Exit code: 0
Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
PASS: import các class training.
NumPy: 2.1.3
Transformers: 5.16.1
Accelerate: 1.14.0
Torch: 2.11.0+cu128
Số GPU: 2
GPU 0 Tesla T4
GPU 1 Tesla T4
CUDA loss: 1.0
CUDA gradient: [0.5, 0.5, 0.5, 0.5]
PASS: CUDA forward/backward.



In [15]:
ROOT = Path("/kaggle/working/vietbud200")
WAV_DIR = ROOT / "subset" / "wavs"

print("CSV 200 mẫu:", (ROOT / "selected_train_200.csv").is_file())
print("Số WAV:", sum(1 for _ in WAV_DIR.glob("*.wav")))
print(
    "Script training:",
    Path(
        "/kaggle/working/Speech-to-Text-PhoWhisper/"
        "kaggle_training/train.py"
    ).is_file(),
)

CSV 200 mẫu: True
Số WAV: 200
Script training: True


In [16]:
import csv
import hashlib
import json
import random
from pathlib import Path

import numpy as np
import soundfile as sf

ROOT = Path("/kaggle/working/vietbud200")
WAV_DIR = (ROOT / "subset" / "wavs").resolve()
CSV_PATH = ROOT / "selected_train_200.csv"
MANIFESTS = ROOT / "manifests"

with CSV_PATH.open(encoding="utf-8-sig", newline="") as f:
    selected = list(csv.DictReader(f, delimiter="|"))

assert len(selected) == 200, f"Nhận {len(selected)} mẫu"

rows = []
seen_ids = set()
seen_hashes = {}
total_seconds = 0

for source in selected:
    relative = Path(source["audio_file"])
    assert (
        not relative.is_absolute()
        and ".." not in relative.parts
        and relative.parent == Path("wavs")
    ), f"Đường dẫn không hợp lệ: {relative}"

    audio_id = relative.name
    text = source["text"].strip()
    path = (WAV_DIR / audio_id).resolve()

    assert path.parent == WAV_DIR and path.is_file(), path
    assert audio_id not in seen_ids, f"ID trùng: {audio_id}"
    assert text, f"Transcript rỗng: {audio_id}"

    info = sf.info(path)
    assert (
        info.frames > 0
        and info.samplerate > 0
        and 0 < info.duration <= 30
    ), f"Audio không hợp lệ hoặc dài quá 30s: {audio_id}"

    audio, rate = sf.read(path, dtype="float32", always_2d=True)
    assert audio.size > 0 and np.isfinite(audio).all(), (
        f"Audio rỗng hoặc có NaN/Inf: {audio_id}"
    )

    digest = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            digest.update(block)
    audio_hash = digest.hexdigest()

    if audio_hash in seen_hashes:
        raise ValueError(
            f"Trùng byte audio: {audio_id} và {seen_hashes[audio_hash]}"
        )

    seen_ids.add(audio_id)
    seen_hashes[audio_hash] = audio_id
    total_seconds += info.duration

    rows.append({
        "id": audio_id,
        "audio_path": str(path),
        "text": text,
        "speaker": source["speaker_name"].strip(),
        "sha256": audio_hash,
    })

# Split cố định, không phụ thuộc thứ tự CSV.
rows.sort(key=lambda row: row["id"])
random.Random(42).shuffle(rows)

train = rows[:180]
validation = rows[180:]

payloads = {
    "train.json": train,
    "validation.json": validation,
    "provenance.json": {
        "seed": 42,
        "train_count": 180,
        "validation_count": 20,
        "selected_csv": str(CSV_PATH),
        "audio_root": str(WAV_DIR),
        "split_method": "sort IDs, shuffle seed 42, split 180/20",
        "technical_only": True,
        "speaker_disjoint": False,
        "possible_speaker_and_content_leakage": True,
        "official_test_used": False,
    },
}

# Cho phép chạy lại nếu nội dung giống hệt;
# không ghi đè một split khác.
encoded = {
    name: json.dumps(data, ensure_ascii=False, indent=2) + "\n"
    for name, data in payloads.items()
}

for name, content in encoded.items():
    target = MANIFESTS / name
    if target.exists():
        assert target.is_file() and target.read_text(
            encoding="utf-8"
        ) == content, f"Manifest cũ khác nội dung: {target}"

MANIFESTS.mkdir(parents=True, exist_ok=True)

for name, content in encoded.items():
    target = MANIFESTS / name
    if not target.exists():
        target.write_text(content, encoding="utf-8")

print("PASS: 200 audio hợp lệ, không trùng byte.")
print("Train:", len(train))
print("Validation:", len(validation))
print("Tổng thời lượng:", round(total_seconds / 60, 2), "phút")
print("Manifests:", MANIFESTS)

PASS: 200 audio hợp lệ, không trùng byte.
Train: 180
Validation: 20
Tổng thời lượng: 8.47 phút
Manifests: /kaggle/working/vietbud200/manifests


In [17]:
import os
import subprocess
import sys
from pathlib import Path

REPO = Path("/kaggle/working/Speech-to-Text-PhoWhisper")
SCRIPT = REPO / "kaggle_training" / "train.py"
MANIFESTS = Path("/kaggle/working/vietbud200/manifests")
OUTPUT = Path("/kaggle/working/vietbud200/smoke-1gpu")

assert SCRIPT.is_file(), "Không tìm thấy script training"
assert (MANIFESTS / "train.json").is_file()
assert (MANIFESTS / "validation.json").is_file()
assert not OUTPUT.exists(), (
    "Output đã tồn tại. Không xóa hoặc ghi đè; gửi log trước."
)

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"

command = [
    sys.executable, "-u", str(SCRIPT), "smoke",
    "--model", "vinai/PhoWhisper-base",
    "--manifest", str(MANIFESTS / "train.json"),
    "--validation", str(MANIFESTS / "validation.json"),
    "--output", str(OUTPUT),
    "--seed", "42",
    "--max-steps", "2",
    "--save-steps", "2",
    "--batch", "1",
    "--accumulation", "1",
    "--workers", "0",
]

print("Bắt đầu smoke test:", flush=True)
print(" ".join(command), flush=True)

subprocess.run(
    command,
    cwd=str(REPO),
    env=env,
    check=True,
)

print("\nPASS: smoke test kết thúc thành công.")
print("Output:", OUTPUT)

Bắt đầu smoke test:
/usr/bin/python3 -u /kaggle/working/Speech-to-Text-PhoWhisper/kaggle_training/train.py smoke --model vinai/PhoWhisper-base --manifest /kaggle/working/vietbud200/manifests/train.json --validation /kaggle/working/vietbud200/manifests/validation.json --output /kaggle/working/vietbud200/smoke-1gpu --seed 42 --max-steps 2 --save-steps 2 --batch 1 --accumulation 1 --workers 0


100%|██████████| 2/2 [00:02<00:00,  1.16it/s]

{'loss': '0.95', 'grad_norm': 'inf', 'learning_rate': '1e-05', 'epoch': '0.005556'}
{'loss': '2.02', 'grad_norm': 'inf', 'learning_rate': '5e-06', 'epoch': '0.01111'}


[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.

100%|██████████| 20/20 [00:04<00:00,  4.73it/s][transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
                                             
100%|██████████| 20/20 [00:04<00:00,  4.73it/s]
                                               
Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'eval_loss': '1.288', 'eval_wer': '0.1421', 'eval_cer': '0.08973', 'eval_substitutions': 21, 'eval_deletions': 0, 'eval_insertions': 5, 'eval_runtime': '4.906', 'eval_samples_per_second': '4.076', 'eval_steps_per_second': '4.076', 'epoch': '0.01111'}



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '8.005', 'train_samples_per_second': '0.25', 'train_steps_per_second': '0.25', 'train_loss': '1.485', 'epoch': '0.01111'}


100%|██████████| 20/20 [00:04<00:00,  4.72it/s]



PASS: smoke test kết thúc thành công.
Output: /kaggle/working/vietbud200/smoke-1gpu


In [18]:
from pathlib import Path
import json
import torch

OUTPUT = Path("/kaggle/working/vietbud200/smoke-1gpu")
CHECKPOINT = OUTPUT / "checkpoint-2"

print("=== FILE CHECKPOINT ===")
print("Checkpoint tồn tại:", CHECKPOINT.is_dir())

if CHECKPOINT.is_dir():
    for p in sorted(CHECKPOINT.iterdir()):
        if p.is_file():
            print(p.name, "|", p.stat().st_size, "bytes")

print("\n=== RUN METADATA ===")
run_path = OUTPUT / "run.json"

if run_path.is_file():
    run = json.loads(run_path.read_text())
    for key in [
        "resolved_revision",
        "world_size",
        "torch",
        "manifest_sha256",
        "validation_sha256",
    ]:
        print(key, ":", run.get(key))

print("\n=== TRAINER STATE ===")
for state_path in [
    CHECKPOINT / "trainer_state.json",
    OUTPUT / "trainer_state.json",
]:
    if state_path.is_file():
        state = json.loads(state_path.read_text())
        print("\nFile:", state_path)
        print("Global step:", state.get("global_step"))
        print("Epoch:", state.get("epoch"))

        for entry in state.get("log_history", []):
            if "loss" in entry or "eval_loss" in entry:
                print(entry)

print("\n=== AMP SCALER ===")
scaler_path = CHECKPOINT / "scaler.pt"

if scaler_path.is_file():
    # Chỉ đọc checkpoint do chính lượt chạy của bạn vừa tạo.
    scaler = torch.load(
        scaler_path,
        map_location="cpu",
        weights_only=True,
    )
    print(scaler)
else:
    print("Không thấy scaler.pt; chưa thể kết luận về AMP từ việc này.")

=== FILE CHECKPOINT ===
Checkpoint tồn tại: True
config.json | 1250 bytes
generation_config.json | 4493 bytes
model.safetensors | 290403936 bytes
optimizer.pt | 580960203 bytes
processor_config.json | 409 bytes
rng_state.pth | 14645 bytes
scaler.pt | 1383 bytes
scheduler.pt | 1465 bytes
tokenizer.json | 3654002 bytes
tokenizer_config.json | 2120 bytes
trainer_state.json | 1470 bytes
training_args.bin | 5393 bytes

=== RUN METADATA ===
resolved_revision : 7ebdb9e88f5cc5271fb88f4d642c82ff9388650e
world_size : 1
torch : 2.11.0+cu128
manifest_sha256 : 55eacf5ddd76d67cc642e6f0ef40df1f5163dbe346774ca85460c511bf1f7378
validation_sha256 : 1e219ce9d70ca182d943db726314404742b2a0777da25b525ed5870b6b6cedca

=== TRAINER STATE ===

File: /kaggle/working/vietbud200/smoke-1gpu/checkpoint-2/trainer_state.json
Global step: 2
Epoch: 0.011111111111111112
{'epoch': 0.005555555555555556, 'grad_norm': inf, 'learning_rate': 1e-05, 'loss': 0.9499627947807312, 'step': 1}
{'epoch': 0.011111111111111112, 'grad_no

In [29]:
from collections import Counter
from pathlib import Path
import torch

path = Path(
    "/kaggle/working/vietbud200/smoke-fp32-1gpu/"
    "checkpoint-2/optimizer.pt"
)

optimizer = torch.load(
    path,
    map_location="cpu",
    weights_only=True,
)

states = optimizer["state"]
steps = Counter()
missing_steps = 0

stats = {
    name: {
        "tensor_count": 0,
        "nonfinite_elements": 0,
        "tensors_with_any_nonzero": 0,
    }
    for name in ["exp_avg", "exp_avg_sq"]
}

for state in states.values():
    if "step" not in state:
        missing_steps += 1
    else:
        step = state["step"]
        if isinstance(step, torch.Tensor):
            assert step.numel() == 1, "Optimizer step không phải scalar"
            step = step.item()
        steps[step] += 1

    for name, counts in stats.items():
        if name not in state:
            continue

        tensor = state[name]
        counts["tensor_count"] += 1
        counts["nonfinite_elements"] += int(
            (~torch.isfinite(tensor)).sum().item()
        )
        counts["tensors_with_any_nonzero"] += int(
            torch.count_nonzero(tensor).item() > 0
        )

print("Số parameter states:", len(states))
print("Phân bố optimizer step:", dict(steps))
print("Số states thiếu step:", missing_steps)

for name, counts in stats.items():
    print(name, ":", counts)

for index, group in enumerate(optimizer["param_groups"]):
    print(f"Learning rate group {index}:", group.get("lr"))

# Giải phóng tham chiếu đến checkpoint lớn khi đã kiểm tra xong.
del states, optimizer

Số parameter states: 245
Phân bố optimizer step: {2.0: 245}
Số states thiếu step: 0
exp_avg : {'tensor_count': 245, 'nonfinite_elements': 0, 'tensors_with_any_nonzero': 245}
exp_avg_sq : {'tensor_count': 245, 'nonfinite_elements': 0, 'tensors_with_any_nonzero': 245}
Learning rate group 0: 0.0
Learning rate group 1: 0.0


In [32]:
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path("/kaggle/working/vietbud200")
OUTPUT = ROOT / "baseline-fp32"

assert not OUTPUT.exists(), (
    "Output baseline đã tồn tại; không ghi đè."
)

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = "0"

subprocess.run(
    [
        sys.executable, "-u",
        str(ROOT / "train_fp32_smoke.py"),
        "evaluate",
        "--model", "vinai/PhoWhisper-base",
        "--revision", "7ebdb9e88f5cc5271fb88f4d642c82ff9388650e",
        "--manifest", str(ROOT / "manifests" / "validation.json"),
        "--output", str(OUTPUT),
        "--workers", "0",
    ],
    env=env,
    check=True,
)

print("Baseline hoàn tất:", OUTPUT)

Loading weights: 100%|██████████| 246/246 [00:00<00:00, 16790.32it/s]
[transformers] Passing `generation_config` together with generation-related arguments=({'begin_suppress_tokens', 'suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
100%|██████████| 20/20 [00:03<00:00,  5.33it/s]


Baseline hoàn tất: /kaggle/working/vietbud200/baseline-fp32


In [33]:
import json
from pathlib import Path

ROOT = Path("/kaggle/working/vietbud200")

for name in ["baseline-fp32", "smoke-fp32-1gpu"]:
    path = ROOT / name / "metrics.json"
    print(f"\n=== {name} ===")
    print(json.dumps(
        json.loads(path.read_text()),
        ensure_ascii=False,
        indent=2,
    ))


=== baseline-fp32 ===
{
  "test_loss": 1.2881582975387573,
  "test_model_preparation_time": 0.0026,
  "test_wer": 0.14207650273224043,
  "test_cer": 0.08972691807542263,
  "test_substitutions": 21,
  "test_deletions": 0,
  "test_insertions": 5,
  "test_runtime": 4.5557,
  "test_samples_per_second": 4.39,
  "test_steps_per_second": 4.39
}

=== smoke-fp32-1gpu ===
{
  "test_loss": 1.1874048709869385,
  "test_wer": 0.14754098360655737,
  "test_cer": 0.09622886866059818,
  "test_substitutions": 20,
  "test_deletions": 0,
  "test_insertions": 7,
  "test_runtime": 3.8118,
  "test_samples_per_second": 5.247,
  "test_steps_per_second": 5.247
}


In [34]:
import hashlib
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path("/kaggle/working/vietbud200")
SCRIPT = ROOT / "train_fp32_smoke.py"
OUTPUT = ROOT / "pilot-fp32-1epoch"

previous = json.loads(
    (ROOT / "smoke-fp32-1gpu" / "run.json").read_text()
)

revision = previous["resolved_revision"]
train_manifest = Path(previous["arguments"]["manifest"])
validation_manifest = Path(previous["arguments"]["validation"])

# Giữ nguyên dữ liệu đã kiểm tra.
for path, key, expected_count in [
    (train_manifest, "manifest_sha256", 180),
    (validation_manifest, "validation_sha256", 20),
]:
    assert path.is_file(), f"Thiếu manifest: {path}"
    assert hashlib.sha256(path.read_bytes()).hexdigest() == previous[key], (
        f"Manifest đã thay đổi: {path}"
    )
    assert len(json.loads(path.read_text())) == expected_count

source = SCRIPT.read_text()
assert source.count("fp16=False") == 1, "Chưa đúng bản script FP32"
assert "fp16=torch.cuda.is_available()" not in source
assert not OUTPUT.exists(), (
    "Output đã tồn tại; không ghi đè hoặc resume tự động."
)

env = os.environ.copy()

for key in list(env):
    if (
        key in {
            "RANK", "LOCAL_RANK", "WORLD_SIZE", "LOCAL_WORLD_SIZE",
            "GROUP_RANK", "ROLE_RANK", "ROLE_WORLD_SIZE",
            "MASTER_ADDR", "MASTER_PORT", "NODE_RANK",
        }
        or key.startswith(("TORCHELASTIC_", "ACCELERATE_"))
    ):
        env.pop(key, None)

env["CUDA_VISIBLE_DEVICES"] = "0"

command = [
    sys.executable, "-u", str(SCRIPT), "train",
    "--model", "vinai/PhoWhisper-base",
    "--revision", revision,
    "--manifest", str(train_manifest),
    "--validation", str(validation_manifest),
    "--output", str(OUTPUT),
    "--seed", "42",
    "--learning-rate", "1e-5",
    "--batch", "1",
    "--accumulation", "1",
    "--workers", "0",
    "--max-steps", "180",
    "--save-steps", "60",
]

print("Pilot FP32: một GPU, khoảng một epoch.", flush=True)
print("Output:", OUTPUT, flush=True)
print("Theo dõi loss/grad_norm; nếu NaN hoặc inf thì dừng.", flush=True)

subprocess.run(command, env=env, check=True)

print("\nPilot hoàn tất.")

Pilot FP32: một GPU, khoảng một epoch.
Output: /kaggle/working/vietbud200/pilot-fp32-1epoch
Theo dõi loss/grad_norm; nếu NaN hoặc inf thì dừng.


  1%|          | 1/180 [00:00<02:32,  1.17it/s]

{'loss': '0.9503', 'grad_norm': '39.12', 'learning_rate': '1e-05', 'epoch': '0.005556'}


  2%|▏         | 3/180 [00:01<01:03,  2.79it/s]

{'loss': '1.91', 'grad_norm': '59.33', 'learning_rate': '9.944e-06', 'epoch': '0.01111'}
{'loss': '1.412', 'grad_norm': '85.71', 'learning_rate': '9.889e-06', 'epoch': '0.01667'}


...

{'eval_loss': '0.2725', 'eval_wer': '0.1311', 'eval_cer': '0.07022', 'eval_substitutions': 21, 'eval_deletions': 1, 'eval_insertions': 2, 'eval_runtime': '3.794', 'eval_samples_per_second': '5.271', 'eval_steps_per_second': '5.271', 'epoch': '1'}



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{'train_runtime': '48.15', 'train_samples_per_second': '3.738', 'train_steps_per_second': '3.738', 'train_loss': '0.2071', 'epoch': '1'}


100%|██████████| 20/20 [00:03<00:00,  5.54it/s]



Pilot hoàn tất.


In [35]:
import json
from pathlib import Path

ROOT = Path("/kaggle/working/vietbud200")
OUTPUT = ROOT / "pilot-fp32-1epoch"

for name in ["baseline-fp32", "pilot-fp32-1epoch"]:
    print(f"\n=== {name} ===")
    metrics = json.loads((ROOT / name / "metrics.json").read_text())
    print(json.dumps(metrics, indent=2))

state = json.loads((OUTPUT / "trainer_state.json").read_text())

print("\n=== TRAINER STATE ===")
print("Global step:", state.get("global_step"))
print("Epoch:", state.get("epoch"))

print("\n=== VALIDATION THEO MỐC ===")
for entry in state.get("log_history", []):
    if "eval_loss" in entry or "train_runtime" in entry:
        print(entry)

print("\nCheckpoint còn giữ:")
for path in sorted(OUTPUT.glob("checkpoint-*")):
    print(path.name)


=== baseline-fp32 ===
{
  "test_loss": 1.2881582975387573,
  "test_model_preparation_time": 0.0026,
  "test_wer": 0.14207650273224043,
  "test_cer": 0.08972691807542263,
  "test_substitutions": 21,
  "test_deletions": 0,
  "test_insertions": 5,
  "test_runtime": 4.5557,
  "test_samples_per_second": 4.39,
  "test_steps_per_second": 4.39
}

=== pilot-fp32-1epoch ===
{
  "test_loss": 0.2725087106227875,
  "test_wer": 0.13114754098360656,
  "test_cer": 0.07022106631989597,
  "test_substitutions": 21,
  "test_deletions": 1,
  "test_insertions": 2,
  "test_runtime": 3.7684,
  "test_samples_per_second": 5.307,
  "test_steps_per_second": 5.307
}

=== TRAINER STATE ===
Global step: 180
Epoch: 1.0

=== VALIDATION THEO MỐC ===
{'epoch': 0.3333333333333333, 'eval_cer': 0.06762028608582575, 'eval_deletions': 1, 'eval_insertions': 1, 'eval_loss': 0.2697628438472748, 'eval_runtime': 3.9223, 'eval_samples_per_second': 5.099, 'eval_steps_per_second': 5.099, 'eval_substitutions': 19, 'eval_wer': 0.1147